# Dataloaders

In [1]:
import os, shutil

## Download BCSS

In [2]:
from google.colab import drive
drive.mount("/content/gdrive/", force_remount=True)

Mounted at /content/gdrive/


In [3]:
!mkdir /content/bcss

Need to add shortcut to [folder](https://drive.google.com/drive/folders/1cW1N4SPL4pi1IbCoF6MJSSlBxDgTtfkb?usp=sharing)

In [4]:
!unzip /content/gdrive/MyDrive/cv_histology/BCSS.zip -d /content/bcss

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: /content/bcss/val_mask/TCGA-A2-A0T0-DX1_xmin72865_ymin59458_MPP-0_448_672_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T0-DX1_xmin72865_ymin59458_MPP-0_896_2240_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T0-DX1_xmin72865_ymin59458_MPP-0_896_3808_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_0_1792_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_0_2240_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_0_448_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_1120_0_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_1120_2016_size224.png  
  inflating: /content/bcss/val_mask/TCGA-A2-A0T2-DX1_xmin63557_ymin56751_MPP-0_1120_4032_size224.png  
  inflating: /content/

# Imports

In [5]:
%%capture
!pip install segmentation-models-pytorch
!pip install torchinfo

In [6]:
# Data handling
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Torch
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp
from torchinfo import summary

# os
import os

# Path
from pathlib import Path

# tqdm
from tqdm.auto import tqdm

# warnings
import warnings
warnings.filterwarnings("ignore")

# Data prepare

In [7]:
!rm -rf ds
!rm -rf images
!rm -rf labels

In [8]:
!mkdir -p /content/ds/train/images/
!mkdir -p /content/ds/train/labels/
!mkdir -p /content/ds/test/

!mkdir /content/images/
!mkdir /content/labels/

In [9]:
import shutil

In [10]:
for p in Path("/content/bcss/train/").glob("*.png"):
    shutil.copy(p, f"/content/images/{p.name}")

for p in Path("/content/bcss/val/").glob("*.png"):
    shutil.copy(p, f"/content/images/{p.name}")

for p in Path("/content/bcss/train_mask/").glob("*.png"):
    shutil.copy(p, f"/content/labels/{p.name}")

for p in Path("/content/bcss/val_mask/").glob("*.png"):
    shutil.copy(p, f"/content/labels/{p.name}")

In [11]:
import random as rnd

In [12]:
rnd.seed(42)

files = sorted([p.name for p in Path("/content/images/").glob("*")])
train_files = rnd.sample(files, k=50)
test_files = rnd.sample(train_files, k=45)

In [13]:
for p in Path("/content/images/").glob("*"):
    if p.name in test_files:
        shutil.move(p, f"/content/ds/test/{p.name}")
    elif p.name in train_files:
        shutil.move(p, f"/content/ds/train/images/")

In [14]:
for p in Path("/content/labels/").glob("*"):
    if p.name in test_files:
        os.remove(p)
    elif p.name in train_files:
        shutil.move(p, f"/content/ds/train/labels/")

# Data load

## Utils

In [15]:
# We define a function to create a list of the paths of the images and masks.
def image_mask_path(image_path: str, mask_path: str):
    IMAGE_PATH = Path(image_path)
    IMAGE_PATH_LIST = sorted(list(IMAGE_PATH.glob("*.png")))

    MASK_PATH = Path(mask_path)
    MASK_PATH_LIST = sorted(list(MASK_PATH.glob("*.png")))

    return IMAGE_PATH_LIST, MASK_PATH_LIST

In [16]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

image_transforms = transforms.Compose([transforms.ToTensor(),
                                       transforms.Normalize(mean = MEAN, std = STD)])

mask_transforms = transforms.Compose([transforms.PILToTensor()])

In [17]:
class CustomImageMaskDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms, mask_transforms):
        self.data = data
        self.image_transforms = image_transforms
        self.mask_transforms = mask_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")
        image = self.image_transforms(image)

        mask_path = self.data.iloc[idx, 1]
        mask = Image.open(mask_path)
        mask = self.mask_transforms(mask)

        return image, mask

In [18]:
class CustomTestDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms):
        self.data = data
        self.image_transforms = image_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")
        image = self.image_transforms(image)

        return image

## General

In [19]:
image_path_train = "/content/ds/train/images"
mask_path_train = "/content/ds/train/labels"

IMAGE_PATH_LIST_TRAIN, MASK_PATH_LIST_TRAIN = image_mask_path(image_path_train,
                                                              mask_path_train)

print(f'Total Images Train: {len(IMAGE_PATH_LIST_TRAIN)}')
print(f'Total Masks Train: {len(MASK_PATH_LIST_TRAIN)}')

Total Images Train: 5
Total Masks Train: 5


In [20]:
VALUES_UNIQUE_TRAIN = []

for i in MASK_PATH_LIST_TRAIN:
    sample = cv2.imread(str(i), cv2.IMREAD_GRAYSCALE)
    uniques = np.unique(sample)
    VALUES_UNIQUE_TRAIN.append(uniques)

FINAL_VALUES_UNIQUE_TRAIN = np.concatenate(VALUES_UNIQUE_TRAIN)
print("Unique values Train:\n")
print(np.unique(FINAL_VALUES_UNIQUE_TRAIN))

Unique values Train:

[0 1 2]


# Model

In [21]:
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

In [22]:
# CUDA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [23]:
def train_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer):

    model.train()

    train_loss = 0.
    train_accuracy = 0.

    for batch, (X,y) in enumerate(dataloader):
        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)
        optimizer.zero_grad()
        logit_mask = model(X)
        loss = loss_fn(logit_mask, y.squeeze())
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp,fp,fn,tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                            target = y.squeeze().cpu().long(),
                                            mode = "multiclass",
                                            num_classes = 21)

        train_accuracy += smp.metrics.accuracy(tp, fp, fn, tn, reduction = "micro").numpy()

    train_loss = train_loss / len(dataloader)
    train_accuracy = train_accuracy / len(dataloader)

    return train_loss, train_accuracy

In [24]:
def train(model:torch.nn.Module, train_dataloader:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer, epochs:int = 10):

    results = {'train_loss':[], 'train_accuracy':[]}

    for epoch in range(epochs):
        train_loss, train_accuracy = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer)

        # print(f'Epoch: {epoch + 1} | ', f'Train Loss: {train_loss:.4f} | ', f'Train Accuracy: {train_accuracy:.4f}')

        results['train_loss'].append(train_loss)
        results['train_accuracy'].append(train_accuracy)

    return results


In [25]:
SEED = 42
EPOCHS = 5
torch.cuda.manual_seed(SEED)
torch.manual_seed(SEED)

In [26]:
!mkdir /content/checkpoints

# U_net star

In [28]:
def predictions_mask(model, test_dataloader: torch.utils.data.DataLoader):
    # checkpoint = torch.load("/content/checkpoints/model.pth")

    model.eval()

    y_pred_mask = []

    with torch.inference_mode():
        for batch,X in enumerate(test_dataloader):
            X = X.to(device = DEVICE, dtype = torch.float32)
            mask_logit = model(X)
            mask_prob = mask_logit.softmax(dim = 1)
            mask_pred = mask_prob.argmax(dim = 1)
            y_pred_mask.append(mask_pred.detach().cpu())

    y_pred_mask = torch.cat(y_pred_mask)

    model.train()

    return y_pred_mask

In [31]:
def predict_upmask(model, dataloader):
    y_pred_mask = []

    model.eval()

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X = X.to(device=DEVICE, dtype = torch.float32)
            y = y.to(device = DEVICE, dtype = torch.long)

            mask_logit = model(X)

            mask_prob = mask_logit.softmax(dim=1)
            mask_pred = mask_prob.argmax(dim=1)

            mask_prob = mask_prob.max(dim=1).values

            y = y.squeeze()

            y[mask_prob > 0.98] = mask_pred[mask_prob > 0.98]

            y_pred_mask.append(y.detach().cpu())

    model.train()

    return torch.cat(y_pred_mask)

In [32]:
SEED = 42
EPOCHS = 5
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()
NUM_CLASSES = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [33]:
ADDED_IMAGES = []

In [34]:
def uptrain_step(image_path_train, mask_path_train, need_create_sample=True, epoch_id=0):
    IMAGE_PATH_LIST_TRAIN, MASK_PATH_LIST_TRAIN = image_mask_path(image_path_train, mask_path_train)

    data_train = pd.DataFrame({'Image':IMAGE_PATH_LIST_TRAIN, 'Mask': MASK_PATH_LIST_TRAIN})
    train_dataset = CustomImageMaskDataset(data_train, image_transforms, mask_transforms)
    train_dataloader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = NUM_WORKERS)

    model = smp.Unet(in_channels=3, classes=NUM_CLASSES, encoder_name="resnet18")

    for param in model.encoder.parameters():
        param.requires_grad = False

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 0.0001)

    train(
        model.to(device = DEVICE),
        train_dataloader,
        loss_fn,
        optimizer,
        EPOCHS
    )

    y_pred_mask = predict_upmask(model, train_dataloader)

    for idx, row in data_train.iterrows():
        cv2.imwrite(str(row[1]), y_pred_mask[idx].squeeze().numpy())

    if need_create_sample:
        image_path_test = "/content/ds/test"
        IMAGE_PATH_LIST_TEST = list(Path(image_path_test).glob("*.png"))

        data_test = pd.DataFrame({'Image':IMAGE_PATH_LIST_TEST})

        test_dataset = CustomTestDataset(data_test, image_transforms)
        test_dataloader = DataLoader(dataset = test_dataset, batch_size = BATCH_SIZE, shuffle = False)

        y_pred_mask = predictions_mask(model, test_dataloader)

        rnd.seed(SEED)

        test_image_idx = rnd.choice(data_test.index)
        test_image_path = data_test.loc[test_image_idx, "Image"]
        test_image_mask = y_pred_mask[test_image_idx]

        shutil.move(test_image_path, "/content/ds/train/images/")
        cv2.imwrite(f"/content/ds/train/labels/{test_image_path.name}", y_pred_mask[test_image_idx].squeeze().numpy())

        ADDED_IMAGES.append(test_image_path.name)

    torch.save(model.state_dict(), f"/content/checkpoints/model{epoch_id}.pth")

In [35]:
rnd.seed(42)
for epoch_id in tqdm(range(15)):
    uptrain_step(image_path_train, mask_path_train, epoch_id=epoch_id, need_create_sample=epoch_id < 5)

  0%|          | 0/15 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth

  0%|          | 0.00/44.7M [00:00<?, ?B/s]
  2%|▏         | 896k/44.7M [00:00<00:05, 8.44MB/s]
  7%|▋         | 3.25M/44.7M [00:00<00:02, 17.5MB/s]
 15%|█▍        | 6.62M/44.7M [00:00<00:01, 25.2MB/s]
 27%|██▋       | 12.2M/44.7M [00:00<00:00, 38.0MB/s]
 44%|████▍     | 19.6M/44.7M [00:00<00:00, 51.5MB/s]
 65%|██████▌   | 29.2M/44.7M [00:00<00:00, 67.8MB/s]
100%|██████████| 44.7M/44.7M [00:00<00:00, 63.5MB/s]


In [36]:
%%capture

checkpoint = torch.load("/content/checkpoints/model14.pth")

model = smp.Unet(encoder_weights = None, classes = 3, encoder_name="resnet18")
model.load_state_dict(checkpoint)
model.to(device = DEVICE)

model.eval()

In [37]:
image_path_val = "/content/images/"
mask_path_val = "/content/labels"

IMAGE_PATH_LIST_VAL, MASK_PATH_LIST_VAL = image_mask_path(image_path_val,
                                                          mask_path_val)

print(f'Total Images Val: {len(IMAGE_PATH_LIST_VAL)}')
print(f'Total Masks Val: {len(MASK_PATH_LIST_VAL)}')

Total Images Val: 36139
Total Masks Val: 36139


In [38]:
data_val = pd.DataFrame({'Image':IMAGE_PATH_LIST_VAL,
                         'Mask':MASK_PATH_LIST_VAL})

In [39]:
val_dataset = CustomImageMaskDataset(data_val, image_transforms,
                                     mask_transforms)

In [40]:
val_dataloader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE,
                            shuffle = True, num_workers = NUM_WORKERS)

In [41]:
test_dice = 0.

with torch.inference_mode():
    for batch, (X, y) in tqdm(enumerate(val_dataloader), total=30):
        if batch == 30:
            break

        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)

        logit_mask = model(X)

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp, fp, fn, tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                                target = y.squeeze().cpu().long(),
                                                mode = "multiclass",
                                                num_classes = 3)

        test_dice += smp.metrics.f1_score(tp, fp, fn, tn, reduction = "micro").numpy()

test_dice /= 30

  0%|          | 0/30 [00:00<?, ?it/s]

In [42]:
test_dice

0.35206496020158135